In [2]:
%%writefile api/index.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Auckland Bus Delay Predictor</title>
<style>
  :root { color-scheme: light dark; --bg:#f6f7f9; --card:#fff; --text:#1c2430; --muted:#5f6b7a; --accent:#0b6bcb; --line:#d9dee5; }
  @media (prefers-color-scheme: dark) { :root { --bg:#12161c; --card:#1b212a; --text:#e8ecf1; --muted:#9aa5b4; --accent:#5aa9ff; --line:#2c3541; } }
  body { margin:0; font-family: system-ui, sans-serif; background:var(--bg); color:var(--text); }
  main { max-width:520px; margin:0 auto; padding:24px 16px 48px; }
  h1 { font-size:1.5rem; margin:0 0 4px; }
  p.sub { color:var(--muted); margin:0 0 20px; }
  .card { background:var(--card); border:1px solid var(--line); border-radius:12px; padding:20px; }
  label { display:block; font-size:.85rem; color:var(--muted); margin:14px 0 4px; }
  select, input, button { width:100%; box-sizing:border-box; padding:10px; font-size:1rem; border-radius:8px; border:1px solid var(--line); background:var(--bg); color:var(--text); }
  button { margin-top:18px; background:var(--accent); color:#fff; border:none; cursor:pointer; font-weight:600; }
  #result { margin-top:20px; display:none; }
  #big { font-size:1.6rem; font-weight:700; }
  #detail, #note, footer { color:var(--muted); font-size:.9rem; margin-top:6px; }
  #note { color:#c0392b; }
</style>
</head>
<body>
<main>
  <h1>Auckland Bus Delay Predictor</h1>
  <p class="sub">Typical delay for a route, direction and time of day, learned from live Auckland Transport data.</p>
  <div class="card">
    <label for="route">Route</label>
    <select id="route"></select>
    <label for="direction">Direction</label>
    <select id="direction"><option value="0">Direction 0</option><option value="1">Direction 1</option></select>
    <label for="hour">Hour of day (Auckland time, 0-23)</label>
    <input id="hour" type="number" min="0" max="23" value="8">
    <label for="stop">Stop number along the trip (optional)</label>
    <input id="stop" type="number" min="1" placeholder="Leave empty for the route's typical stop">
    <button id="go">Predict delay</button>
    <div id="result">
      <div id="big"></div>
      <div id="detail"></div>
      <div id="note"></div>
    </div>
  </div>
  <footer id="foot"></footer>
</main>
<script>
const $ = id => document.getElementById(id);

async function init() {
  const routes = await (await fetch("/routes")).json();
  routes.forEach(r => { const o = document.createElement("option"); o.value = r; o.textContent = r; $("route").appendChild(o); });
  const h = await (await fetch("/health")).json();
  $("foot").textContent = "Model trained on " + h.trained_rows.toLocaleString() + " live readings. Typical error is about " + Math.round(h.mae_seconds / 60) + " minutes.";
}

function describe(sec) {
  const m = Math.round(Math.abs(sec) / 60);
  if (m === 0) return "about on time";
  return "about " + m + " min " + (sec > 0 ? "late" : "early");
}

$("go").addEventListener("click", async () => {
  const p = new URLSearchParams({ route: $("route").value, direction: $("direction").value, hour: $("hour").value });
  if ($("stop").value) p.set("stop_sequence", $("stop").value);
  const res = await fetch("/predict?" + p);
  const data = await res.json();
  $("result").style.display = "block";
  if (!res.ok) { $("big").textContent = "Error"; $("detail").textContent = JSON.stringify(data.detail); $("note").textContent = ""; return; }
  $("big").textContent = describe(data.predicted_delay_seconds);
  $("detail").textContent = "Predicted delay: " + data.predicted_delay_seconds + " s at stop " + data.stop_sequence + " (typical error about " + Math.round(data.typical_error_seconds) + " s).";
  $("note").textContent = data.note || "";
});

init();
</script>
</body>
</html>

Writing api/index.html


In [4]:
%%writefile api/app.py
import json
from pathlib import Path
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse

BASE = Path(__file__).parent
model = joblib.load(BASE / "model.joblib")
meta = json.loads((BASE / "meta.json").read_text())
ROUTES = meta["route_names"]

app = FastAPI(title="Auckland Bus Delay Predictor")

@app.get("/", include_in_schema=False)
def home():
    return FileResponse(BASE / "index.html")

@app.get("/health")
def health():
    return {"status": "ok", "trained_rows": meta["trained_rows"], "mae_seconds": meta["mae_seconds"]}

@app.get("/routes")
def routes():
    return ROUTES

@app.get("/predict")
def predict(route: str, direction: int = 0, hour: int = 8, stop_sequence: int | None = None):
    if route not in ROUTES:
        raise HTTPException(status_code=404, detail="Unknown route")
    if direction not in (0, 1):
        raise HTTPException(status_code=422, detail="direction must be 0 or 1")
    if not 0 <= hour <= 23:
        raise HTTPException(status_code=422, detail="hour must be 0-23")
    seq = stop_sequence if stop_sequence is not None else int(meta["median_stop_seq"][route])
    X = pd.DataFrame([{"route_code": ROUTES.index(route), "direction_id": direction,
                       "hour": hour, "stop_sequence": seq}])
    delay = float(model.predict(X)[0])
    note = None if hour in meta["hours_seen"] else "No training data for this hour, so treat this as a rough guess."
    return {"route": route, "direction": direction, "hour": hour, "stop_sequence": seq,
            "predicted_delay_seconds": round(delay),
            "typical_error_seconds": meta["mae_seconds"], "note": note}

Overwriting api/app.py


In [5]:
import pandas as pd, json

trips = pd.read_csv("gtfs/trips.txt", usecols=["route_id", "direction_id", "trip_headsign"], dtype={"route_id": str})
routes = pd.read_csv("gtfs/routes.txt", usecols=["route_id", "route_short_name", "route_long_name", "route_type"],
                     dtype={"route_id": str, "route_short_name": str, "route_long_name": str})
t = trips.merge(routes, on="route_id")
t = t[t["route_type"] == 3]

info = {}
for short, g in t.groupby("route_short_name"):
    dirs = {}
    for d_id, gd in g.dropna(subset=["direction_id"]).groupby("direction_id"):
        hs = gd["trip_headsign"].dropna()
        dirs[str(int(d_id))] = hs.mode().iat[0] if len(hs) else None
    ln = g["route_long_name"].dropna()
    info[str(short)] = {"name": ln.mode().iat[0] if len(ln) else "", "dirs": dirs}

names = json.load(open("api/meta.json"))["route_names"]
missing = [r for r in names if r not in info]
json.dump(info, open("api/routes_info.json", "w"))
print(len(info), "routes named. Model routes with no name:", len(missing), missing[:8])

196 routes named. Model routes with no name: 0 []


In [6]:
%%writefile api/app.py
import json
from pathlib import Path
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse

BASE = Path(__file__).parent
model = joblib.load(BASE / "model.joblib")
meta = json.loads((BASE / "meta.json").read_text())
info_path = BASE / "routes_info.json"
INFO = json.loads(info_path.read_text()) if info_path.exists() else {}
ROUTES = meta["route_names"]
HOURS_SEEN = set(meta["hours_seen"])

app = FastAPI(title="Auckland Bus Delay Predictor")

def check(route, direction):
    if route not in ROUTES:
        raise HTTPException(status_code=404, detail="Unknown route")
    if direction not in (0, 1):
        raise HTTPException(status_code=422, detail="direction must be 0 or 1")

@app.get("/", include_in_schema=False)
def home():
    return FileResponse(BASE / "index.html")

@app.get("/health")
def health():
    return {"status": "ok", "trained_rows": meta["trained_rows"], "mae_seconds": meta["mae_seconds"]}

@app.get("/routes")
def routes():
    return [{"route": r, "name": INFO.get(r, {}).get("name", ""), "dirs": INFO.get(r, {}).get("dirs", {})} for r in ROUTES]

@app.get("/profile")
def profile(route: str, direction: int = 0, stop_sequence: int | None = None):
    check(route, direction)
    seq = stop_sequence if stop_sequence is not None else int(meta["median_stop_seq"][route])
    X = pd.DataFrame({"route_code": [ROUTES.index(route)] * 24, "direction_id": [direction] * 24,
                      "hour": list(range(24)), "stop_sequence": [seq] * 24})
    preds = model.predict(X)
    return {"route": route, "direction": direction, "stop_sequence": seq,
            "typical_error_seconds": meta["mae_seconds"], "trained_rows": meta["trained_rows"],
            "hours": [{"hour": h, "delay": round(float(p)), "seen": h in HOURS_SEEN} for h, p in zip(range(24), preds)]}

@app.get("/predict")
def predict(route: str, direction: int = 0, hour: int = 8, stop_sequence: int | None = None):
    check(route, direction)
    if not 0 <= hour <= 23:
        raise HTTPException(status_code=422, detail="hour must be 0-23")
    seq = stop_sequence if stop_sequence is not None else int(meta["median_stop_seq"][route])
    X = pd.DataFrame([{"route_code": ROUTES.index(route), "direction_id": direction, "hour": hour, "stop_sequence": seq}])
    delay = float(model.predict(X)[0])
    note = None if hour in HOURS_SEEN else "No training data for this hour, so treat this as a rough guess."
    return {"route": route, "direction": direction, "hour": hour, "stop_sequence": seq,
            "predicted_delay_seconds": round(delay), "typical_error_seconds": meta["mae_seconds"], "note": note}

Overwriting api/app.py


In [7]:
%%writefile api/index.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>How late is my bus?</title>
<link href="https://fonts.googleapis.com/css2?family=Barlow+Condensed:wght@600;700&family=Figtree:wght@400;600&display=swap" rel="stylesheet">
<style>
:root{color-scheme:light dark;--bg:#EEF4F5;--panel:#fff;--ink:#10243A;--muted:#5B6B7B;--line:#D5E0E3;--badge:#F4B73B;--badge-ink:#10243A;--ok:#1E8A5A;--early:#2D6BB5;--slight:#A8670B;--late:#C8452F;--band:rgba(16,36,58,.07);--unseen:#8795A3}
@media(prefers-color-scheme:dark){:root{--bg:#0C1824;--panel:#122234;--ink:#EAF1F5;--muted:#9FB0BF;--line:#243A50;--ok:#4CC28A;--early:#6DB2FF;--slight:#F2B24C;--late:#FF7A63;--band:rgba(255,255,255,.07);--unseen:#7F90A0}}
body{margin:0;background:var(--bg);color:var(--ink);font:400 16px/1.5 Figtree,system-ui,sans-serif}
main{max-width:880px;margin:0 auto;padding:32px 16px 56px}
h1{font:700 clamp(2.2rem,6vw,3.4rem)/1 "Barlow Condensed","Arial Narrow",sans-serif;margin:0 0 8px}
.sub{color:var(--muted);max-width:60ch;margin:0 0 24px}
.panel{background:var(--panel);border:1px solid var(--line);border-radius:16px;overflow:hidden}
.panel>section{padding:20px 24px;border-top:1px solid var(--line)}
.panel>section:first-child{border-top:0}
.controls{display:grid;grid-template-columns:minmax(0,1fr) minmax(0,1.3fr);gap:16px 24px}
@media(max-width:640px){.controls{grid-template-columns:1fr}.panel>section{padding:16px}}
label,.lbl{display:block;font-size:.875rem;color:var(--muted);margin-bottom:6px}
select{width:100%;padding:11px 12px;font:inherit;color:var(--ink);background:var(--bg);border:1px solid var(--line);border-radius:10px}
.seg{display:flex;gap:6px}
.seg button{flex:1;min-width:0;padding:11px 12px;font:inherit;color:var(--ink);background:var(--bg);border:1px solid var(--line);border-radius:10px;cursor:pointer;text-align:left;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.seg button[aria-checked=true]{background:var(--ink);color:var(--panel);border-color:var(--ink)}
:focus-visible{outline:3px solid var(--early);outline-offset:2px}
.readout{display:flex;align-items:center;gap:22px;flex-wrap:wrap}
.badge{min-width:104px;padding:6px 16px;background:var(--badge);color:var(--badge-ink);border-radius:14px;text-align:center;font:700 3.4rem/1.05 "Barlow Condensed","Arial Narrow",sans-serif}
#verdict{font:700 clamp(2.4rem,8vw,4.2rem)/1 "Barlow Condensed","Arial Narrow",sans-serif}
.ok{color:var(--ok)}.early{color:var(--early)}.slight{color:var(--slight)}.late{color:var(--late)}
#where{color:var(--muted);margin-top:6px}
.timehead{display:flex;justify-content:space-between;align-items:baseline}
#timeLabel{font:700 1.6rem "Barlow Condensed","Arial Narrow",sans-serif}
input[type=range]{width:100%;accent-color:var(--ink);margin:4px 0}
#note{color:var(--slight);min-height:1.5em;margin:6px 0 0}
#chart{display:block;width:100%;height:300px;touch-action:pan-y;cursor:crosshair}
svg text{font:12px Figtree,system-ui,sans-serif;fill:var(--muted)}
.legend{display:flex;gap:20px;flex-wrap:wrap;color:var(--muted);font-size:.85rem;margin-top:8px}
.legend i{display:inline-block;width:24px;border-top:3px solid var(--ink);vertical-align:middle;margin-right:6px}
.legend i.d{border-top:3px dashed var(--unseen)}
.legend i.b{height:10px;border:0;background:var(--band);border-radius:3px}
footer{color:var(--muted);font-size:.85rem;margin-top:14px;max-width:70ch}
</style>
</head>
<body>
<main>
<h1>How late is my bus?</h1>
<p class="sub">Typical delays on Auckland buses, learned from live Auckland Transport data. Pick a route, then slide through the day.</p>
<div class="panel">
  <section class="controls">
    <div><label for="route">Route</label><select id="route"></select></div>
    <div><span class="lbl" id="dl">Heading</span><div class="seg" id="dirs" role="radiogroup" aria-labelledby="dl"></div></div>
  </section>
  <section class="readout">
    <div class="badge" id="badge">-</div>
    <div><div id="verdict">Loading</div><div id="where"></div></div>
  </section>
  <section>
    <div class="timehead"><label for="hour">Time of day</label><span id="timeLabel"></span></div>
    <input id="hour" type="range" min="0" max="23" step="1" value="8">
    <p id="note"></p>
  </section>
  <section id="chartwrap">
    <svg id="chart" role="img" aria-label="Predicted delay across the day"></svg>
    <div class="legend"><span><i></i>Hours we have recorded</span><span><i class="d"></i>Not recorded yet (estimate)</span><span><i class="b"></i>Typical miss of the model</span></div>
  </section>
</div>
<footer id="foot"></footer>
</main>
<script>
const $ = id => document.getElementById(id);
const COL = { ok: "var(--ok)", early: "var(--early)", slight: "var(--slight)", late: "var(--late)" };
const nowHour = () => parseInt(new Intl.DateTimeFormat("en-NZ", { hour: "numeric", hour12: false, timeZone: "Pacific/Auckland" }).format(new Date()), 10) % 24;
const clock = h => (h % 12 === 0 ? 12 : h % 12) + ":00 " + (h < 12 ? "am" : "pm");
const tick = h => (h % 12 === 0 ? 12 : h % 12) + (h < 12 ? "a" : "p");
const S = { routes: {}, route: "83", dir: 0, hour: nowHour(), prof: null };
const G = { W: 760, L: 78, R: 16 };

function verdict(sec) {
  const m = Math.round(Math.abs(sec) / 60);
  if (Math.abs(sec) < 60) return { text: "On time", cls: "ok" };
  return sec > 0 ? { text: m + " min late", cls: sec >= 300 ? "late" : "slight" } : { text: m + " min early", cls: "early" };
}
function dirs() {
  const r = S.routes[S.route], d = r.dirs || {};
  const ks = Object.keys(d).filter(k => d[k]);
  return ks.length ? ks.map(k => ({ id: +k, label: "to " + d[k] })) : [{ id: 0, label: "Direction 0" }, { id: 1, label: "Direction 1" }];
}
function fail() {
  $("verdict").textContent = "No predictions";
  $("verdict").className = "late";
  $("where").textContent = "Can't reach the prediction service. Check that the server is running, then reload.";
}
function drawDirs() {
  const box = $("dirs"), ds = dirs();
  box.replaceChildren();
  if (!ds.some(d => d.id === S.dir)) S.dir = ds[0].id;
  ds.forEach(d => {
    const b = document.createElement("button");
    b.type = "button"; b.setAttribute("role", "radio"); b.setAttribute("aria-checked", d.id === S.dir);
    b.textContent = d.label; b.title = d.label;
    b.onclick = () => { S.dir = d.id; load(); };
    box.appendChild(b);
  });
}
async function load() {
  drawDirs();
  $("chartwrap").style.opacity = .5;
  try {
    const r = await fetch("/profile?route=" + encodeURIComponent(S.route) + "&direction=" + S.dir);
    if (!r.ok) throw new Error(r.status);
    S.prof = await r.json();
    render();
  } catch (e) { fail(); }
  $("chartwrap").style.opacity = 1;
}
function render() {
  const p = S.prof, pt = p.hours[S.hour], v = verdict(pt.delay), d = dirs().find(x => x.id === S.dir);
  $("badge").textContent = S.route;
  $("verdict").textContent = v.text; $("verdict").className = v.cls;
  $("where").textContent = "Route " + S.route + ", " + d.label + ", around " + clock(S.hour);
  $("timeLabel").textContent = clock(S.hour);
  $("note").textContent = pt.seen ? "" : "No buses recorded at this hour yet, so this is an estimate.";
  $("foot").textContent = "Based on " + p.trained_rows.toLocaleString() + " live readings from Auckland Transport. The model's typical miss is about " + Math.round(p.typical_error_seconds / 60) + " min, so treat this as a guide, not a timetable.";
  chart(v.cls);
}
function chart(cls) {
  const svg = $("chart"), W = Math.max(300, Math.round(svg.getBoundingClientRect().width)), H = 300;
  const L = W < 520 ? 68 : 78, R = 16, T = 30, B = 32;
  G.W = W; G.L = L; G.R = R;
  const p = S.prof, err = p.typical_error_seconds / 60, v = p.hours.map(h => h.delay / 60);
  const lo0 = Math.min(-2, ...v.map(x => x - err)), hi0 = Math.max(6, ...v.map(x => x + err));
  const step = (hi0 - lo0) > 16 ? 5 : 2, lo = Math.floor(lo0 / step) * step, hi = Math.ceil(hi0 / step) * step;
  const X = h => L + h * (W - L - R) / 23, Y = m => T + (hi - m) * (H - T - B) / (hi - lo);
  let s = "";
  [[7, 9, "Morning peak"], [16, 18, "Evening peak"]].forEach(([a, b, t]) => {
    s += `<rect x="${X(a)}" y="${T}" width="${X(b) - X(a)}" height="${H - T - B}" fill="var(--band)"/><text x="${X(a)}" y="${T - 10}">${t}</text>`;
  });
  for (let g = lo; g <= hi; g += step) {
    const lab = g === 0 ? "on time" : g > 0 ? g + " min late" : (-g) + " min early";
    s += `<line x1="${L}" x2="${W - R}" y1="${Y(g)}" y2="${Y(g)}" stroke="var(--line)" stroke-width="${g === 0 ? 2 : 1}"/><text x="${L - 8}" y="${Y(g) + 4}" text-anchor="end">${lab}</text>`;
  }
  for (let h = 0; h < 24; h += 3) s += `<text x="${X(h)}" y="${H - 8}" text-anchor="middle">${tick(h)}</text>`;
  const up = v.map((x, i) => `${X(i)},${Y(x + err)}`), dn = v.map((x, i) => `${X(i)},${Y(x - err)}`).reverse();
  s += `<polygon points="${up.concat(dn).join(" ")}" fill="var(--band)"/>`;
  let solid = "", dash = "";
  for (let i = 1; i < 24; i++) {
    const seg = `M${X(i - 1)} ${Y(v[i - 1])}L${X(i)} ${Y(v[i])}`;
    if (p.hours[i].seen && p.hours[i - 1].seen) solid += seg; else dash += seg;
  }
  s += `<path d="${dash}" fill="none" stroke="var(--unseen)" stroke-width="2.5" stroke-dasharray="6 6" stroke-linecap="round"/>`;
  s += `<path d="${solid}" fill="none" stroke="var(--ink)" stroke-width="3" stroke-linecap="round"/>`;
  const cx = X(S.hour), cy = Y(v[S.hour]);
  s += `<line x1="${cx}" x2="${cx}" y1="${T}" y2="${H - B}" stroke="var(--ink)" stroke-dasharray="2 3"/><circle cx="${cx}" cy="${cy}" r="7" fill="${COL[cls]}" stroke="var(--panel)" stroke-width="3"/>`;
  svg.setAttribute("viewBox", `0 0 ${W} ${H}`);
  svg.setAttribute("aria-label", "Predicted delay for route " + S.route + " across the day. At " + clock(S.hour) + ": " + verdict(p.hours[S.hour].delay).text + ".");
  svg.innerHTML = s;
}
function pick(e) {
  const svg = $("chart"), r = svg.getBoundingClientRect(), x = (e.clientX - r.left) * (G.W / r.width);
  S.hour = Math.max(0, Math.min(23, Math.round((x - G.L) * 23 / (G.W - G.L - G.R))));
  $("hour").value = S.hour; render();
}
$("chart").addEventListener("pointerdown", e => { $("chart").setPointerCapture(e.pointerId); if (S.prof) pick(e); });
$("chart").addEventListener("pointermove", e => { if (e.buttons && S.prof) pick(e); });
$("hour").addEventListener("input", e => { S.hour = +e.target.value; if (S.prof) render(); });
$("route").addEventListener("change", e => { S.route = e.target.value; load(); });
window.addEventListener("resize", () => { if (S.prof) render(); });

(async function init() {
  try {
    const list = await (await fetch("/routes")).json();
    list.forEach(r => { S.routes[r.route] = r; });
    const sel = $("route"), col = new Intl.Collator(undefined, { numeric: true });
    Object.keys(S.routes).sort(col.compare).forEach(k => {
      const o = document.createElement("option");
      o.value = k; o.textContent = k + (S.routes[k].name ? ": " + S.routes[k].name : "");
      sel.appendChild(o);
    });
    if (!S.routes[S.route]) S.route = sel.options[0].value;
    sel.value = S.route; $("hour").value = S.hour;
    await load();
  } catch (e) { fail(); }
})();
</script>
</body>
</html>

Overwriting api/index.html


In [8]:
from pathlib import Path

p = Path("api/index.html")
h = p.read_text(encoding="utf-8")

edits = [
 # lighter error band, amber peak bands (light and dark)
 (r'''--band:rgba(16,36,58,.07);''', r'''--band:rgba(16,36,58,.05);--peak:rgba(244,183,59,.17);'''),
 (r'''--band:rgba(255,255,255,.07);''', r'''--band:rgba(255,255,255,.06);--peak:rgba(244,183,59,.14);'''),
 (r'''height="${H - T - B}" fill="var(--band)"/>''', r'''height="${H - T - B}" fill="var(--peak)"/>'''),
 # wrapping direction buttons and faded estimates
 (r'''</style>''', r'''.seg button{white-space:normal;overflow:visible;text-overflow:clip;line-height:1.25}
#verdict.est{opacity:.5}
</style>'''),
 (r'''>Heading</span>''', r'''>Direction</span>'''),
 # destination-only labels
 (r'''function dirs() {''', r'''const dest = hs => { const p = hs.split(/ to /i); return p[p.length - 1].trim(); };
function dirs() {'''),
 (r'''label: "to " + d[k] }''', r'''label: "to " + dest(d[k]) }'''),
 # no "50A: 50A"
 (r'''o.textContent = k + (S.routes[k].name ? ": " + S.routes[k].name : "");''',
  r'''const nm = S.routes[k].name; o.textContent = nm && nm !== k ? k + ": " + nm : k;'''),
 # open on an hour that has data
 (r'''S.prof = await r.json();''', r'''S.prof = await r.json();
    if (!S.settled) { S.settled = true; if (!S.prof.hours[S.hour].seen) { const f = S.prof.hours.find(x => x.seen); if (f) { S.hour = f.hour; $("hour").value = f.hour; } } }'''),
 # fade the verdict when it is an estimate
 (r'''$("verdict").textContent = v.text; $("verdict").className = v.cls;''',
  r'''$("verdict").textContent = v.text; $("verdict").className = v.cls; if (!pt.seen) $("verdict").classList.add("est");'''),
]

for old, new in edits:
    assert h.count(old) == 1, "No unique match for: " + old[:60]
    h = h.replace(old, new)

p.write_text(h, encoding="utf-8")
print("Patched", len(edits), "edits")

Patched 10 edits


In [9]:
from pathlib import Path

p = Path("api/index.html")
h = p.read_text(encoding="utf-8")

edits = [
 (r'''.legend{display:flex;gap:20px;flex-wrap:wrap;color:var(--muted);font-size:.85rem;margin-top:8px}''',
  r'''.legend{display:flex;gap:6px 16px;flex-wrap:wrap;color:var(--muted);font-size:.8rem;margin-top:8px}'''),
 (r'''.legend i{display:inline-block;width:24px;''', r'''.legend i{display:inline-block;width:18px;'''),
 (r'''#note{color:var(--slight);min-height:1.5em;margin:6px 0 0}''',
  r'''#note{color:var(--slight);margin:6px 0 0}#note:empty{display:none}'''),
 (r'''input[type=range]{width:100%;accent-color:var(--ink);margin:4px 0}''',
  r'''input[type=range]{width:100%;height:32px;accent-color:var(--ink);margin:0}'''),
 (r'''return p[p.length - 1].trim(); };''', r'''return p[p.length - 1].trim().replace(/\bVia\b/g, "via"); };'''),
]

for old, new in edits:
    assert h.count(old) == 1, "No unique match for: " + old[:60]
    h = h.replace(old, new)

p.write_text(h, encoding="utf-8")
print("Patched", len(edits), "edits")

Patched 5 edits


In [10]:
%%writefile .gitignore
.env
*.db
gtfs/
venv/
.venv/
__pycache__/
.ipynb_checkpoints/
*.pem

Overwriting .gitignore


In [17]:
import os, pathlib
from dotenv import load_dotenv

load_dotenv(override=True)
key = os.getenv("AT_API_KEY")
print("Key loaded:", bool(key) and len(key) > 30)

hits = []
for f in pathlib.Path(".").rglob("*"):
    if (f.is_file() and f.suffix in {".ipynb", ".py", ".md", ".txt", ".json", ".html"}
            and not {".git", "gtfs", "venv"} & set(f.parts) and f.name != ".env"):
        try:
            if key and key in f.read_text(errors="ignore"):
                hits.append(str(f))
        except Exception:
            pass
print("Files containing your key:", hits)

Key loaded: True
Files containing your key: []


In [13]:
import sqlite3, pandas as pd
c = sqlite3.connect("bus_delays_server.db")
print(pd.read_sql("""
  SELECT COUNT(*) AS rows,
         COUNT(DISTINCT polled_at) AS polls,
         datetime(MIN(polled_at),'unixepoch','+13 hours') AS first_nz,
         datetime(MAX(polled_at),'unixepoch','+13 hours') AS last_nz
  FROM trip_delays""", c))
print(pd.read_sql("""
  SELECT strftime('%H', polled_at, 'unixepoch', '+13 hours') AS hour_nz, COUNT(DISTINCT polled_at) AS polls
  FROM trip_delays GROUP BY hour_nz ORDER BY hour_nz""", c).T)

     rows  polls             first_nz              last_nz
0  106138    218  2026-10-28 16:19:28  2026-10-28 19:56:02
          0   1   2   3
hour_nz  16  17  18  19
polls    41  60  60  57


In [14]:
import sqlite3, pandas as pd, numpy as np

def load(db):
    df = pd.read_sql("SELECT * FROM trip_delays", sqlite3.connect(db))
    df["source"] = db
    return df

raw = pd.concat([load("bus_delays.db"), load("bus_delays_server.db")], ignore_index=True)
raw = raw.drop_duplicates(subset=["trip_id", "start_date", "feed_ts"])

trips = pd.read_csv("gtfs/trips.txt", usecols=["trip_id", "route_id"], dtype=str).rename(columns={"route_id": "static_route_id"})
routes = pd.read_csv("gtfs/routes.txt", usecols=["route_id", "route_short_name", "route_type"],
                     dtype={"route_id": str, "route_short_name": str}).rename(columns={"route_id": "static_route_id"})
tr = trips.merge(routes, on="static_route_id", how="left")

df = raw.merge(tr, on="trip_id", how="left")
print("Trip match rate by source:")
print(df.groupby("source")["route_short_name"].apply(lambda s: round(s.notna().mean(), 3)))

df = df[df["route_type"] == 3].copy()
df["age_s"] = df["polled_at"] - df["feed_ts"]
df = df[(df["age_s"] <= 120) & (df["schedule_rel"].fillna(0) == 0) & df["trip_delay"].between(-900, 1800)].copy()
df["nz_time"] = pd.to_datetime(df["feed_ts"], unit="s", utc=True).dt.tz_convert("Pacific/Auckland")
df["hour"] = df["nz_time"].dt.hour
df["date"] = df["nz_time"].dt.date

print("Clean bus rows:", len(df))
print(df.groupby(["hour", "date"]).size().unstack("date").fillna(0).astype(int))

Trip match rate by source:
source
bus_delays.db           0.988
bus_delays_server.db    0.997
Name: route_short_name, dtype: float64
Clean bus rows: 250827
date  2026-10-22  2026-10-23  2026-10-28
hour                                    
8              0       21018           0
9              0       25001           0
10             0       21837           0
11             0       21786           0
12             0       21811           0
13             0       21804           0
14             0        8706           0
16             0           0       28882
17             0           0       30834
18             0           0       28668
19             0           0       19679
20           801           0           0


In [15]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

d = df[["route_short_name", "direction_id", "hour", "stop_sequence", "trip_delay", "trip_id"]].dropna().copy()
cat = d["route_short_name"].astype("category")
cats = list(cat.cat.categories)
d["route_code"] = cat.cat.codes
feats = ["route_code", "direction_id", "hour", "stop_sequence"]

tr_i, te_i = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(d, groups=d["trip_id"]))
train, test = d.iloc[tr_i], d.iloc[te_i]

gmed = train["trip_delay"].median()
rmed = train.groupby("route_short_name")["trip_delay"].median()
rhmed = train.groupby(["route_short_name", "hour"])["trip_delay"].median()

b1 = np.full(len(test), gmed)
b2 = test["route_short_name"].map(rmed).fillna(gmed)
keys = pd.MultiIndex.from_frame(test[["route_short_name", "hour"]])
b3 = pd.Series(rhmed.reindex(keys).values, index=test.index).fillna(b2)

model = HistGradientBoostingRegressor(categorical_features=[True, False, False, False],
                                      max_iter=200, learning_rate=0.08, random_state=42)
model.fit(train[feats], train["trip_delay"])
pred = model.predict(test[feats])

y = test["trip_delay"]
print("Train rows:", len(train), "| Test rows:", len(test))
print("MAE global median     :", round(mean_absolute_error(y, b1), 1))
print("MAE route median      :", round(mean_absolute_error(y, b2), 1))
print("MAE route+hour median :", round(mean_absolute_error(y, b3), 1))
print("MAE gradient boosting :", round(mean_absolute_error(y, pred), 1))

Train rows: 188484 | Test rows: 62343
MAE global median     : 172.4
MAE route median      : 163.2
MAE route+hour median : 173.6
MAE gradient boosting : 141.4


In [16]:
import json, joblib

final = HistGradientBoostingRegressor(categorical_features=[True, False, False, False],
                                      max_iter=200, learning_rate=0.08, random_state=42)
final.fit(d[feats], d["trip_delay"])

counts = d["hour"].value_counts()
meta = {
    "route_names": cats,
    "median_stop_seq": d.groupby("route_short_name")["stop_sequence"].median().to_dict(),
    "hours_seen": sorted(int(h) for h, n in counts.items() if n >= 2000),
    "trained_rows": int(len(d)),
    "mae_seconds": round(float(mean_absolute_error(y, pred)), 1),
}
joblib.dump(final, "api/model.joblib")
json.dump(meta, open("api/meta.json", "w"))
print("Saved. Routes:", len(cats), "| hours recorded:", meta["hours_seen"], "| MAE:", meta["mae_seconds"])

Saved. Routes: 189 | hours recorded: [8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19] | MAE: 141.4
